# 알콩약콩 | 학습 전 baseline 평가

두 파일을 같은 폴더에 두고 이 노트북을 위에서부터 실행하세요. Colab에서는 두 파일을 `/content`에 업로드하면 됩니다.

기본값은 **합성 예시 데모**입니다. GPU와 모델 다운로드 없이 입력 형식·표·그래프를 확인합니다. 데모의 ROUGE-L은 합성 답변으로 계산하며 BERTScore와 Faithfulness는 미측정으로 표시합니다.

실제 평가에서는 GPU 런타임을 선택하고 설치 셀의 두 번째 설치 명령을 실행하세요. 설치 후 런타임 재시작이 필요할 수 있습니다.

In [ ]:
%pip install -q pandas matplotlib
# 실제 평가 시 주석을 해제하세요. CUDA PyTorch가 준비된 환경을 사용합니다.
# %pip install -U "transformers>=5.2,<6" accelerate bitsandbytes bert-score pillow torchvision

## 1. 데이터와 실행 옵션

CSV 또는 JSONL에서 한 행이 질문 하나입니다.

| 필드 | 의미 |
|---|---|
| `id` | 고유 샘플 ID |
| `category` | 병용금기, 임부금기, 특정연령대금기, 기타 |
| `question` | 사용자 질문 |
| `evidence` | 모델에 제공할 DUR 근거. 근거가 없는 경우 빈 문자열 |
| `reference` | 검토된 모범답안. 지표 계산에만 사용 |
| `response_type` | 기대 응답 유형. 설명 / 추가 질문 / 판단 보류. 판단 보류 성공률 산출에 사용 |

데이터 준비 후 `DEMO=False`로 바꾸고 `DATASET`의 경로를 지정하세요. `LIMIT=2`로 모델 동작을 확인하고 전체 평가에는 `None`을 사용하세요. Validation과 Test는 별도 파일로 관리하세요.

In [ ]:
from datetime import datetime
from pathlib import Path
import json

from IPython.display import Image, display
from baseline_eval import evaluate, MODELS, JUDGE, ENCODER

DEMO = True
DATASET = Path("validation.jsonl")
SPLIT = "validation"
LIMIT = None
CANDIDATES = MODELS.copy()
JUDGE_MODEL = JUDGE
BERT_ENCODER = ENCODER
MAX_NEW_TOKENS = 256
JUDGE_TOKENS = 768
REVISIONS = {}  # 재현 시 run_config.json에서 revisions를 복사할 수 있습니다.

tag = "demo" if DEMO else SPLIT
OUTPUT = Path("baseline_results") / f"{tag}_{datetime.now():%Y%m%d_%H%M%S_%f}"
if DEMO:
    examples = [
        {"id": "demo1", "category": "병용금기", "question": "가상 성분 A와 B의 기록은?",
         "evidence": "가상 기록에서 A와 B는 병용금기로 표시됨.",
         "reference": "가상 기록에서 A와 B는 병용금기로 표시됩니다."},
        {"id": "demo2", "category": "임부금기", "question": "가상 성분 C의 기록은?",
         "evidence": "가상 기록에 임부금기 항목이 있음.",
         "reference": "가상 기록에 임부금기 항목이 있습니다."},
        {"id": "demo3", "category": "특정연령대금기", "question": "가상 성분 D의 기록은?",
         "evidence": "가상 기록에 연령 조건이 있음.",
         "reference": "가상 기록에 연령 조건이 있습니다."},
        {"id": "demo4", "category": "기타", "response_type": "판단 보류", "question": "이 약에 해당하는 기록이 있나요?",
         "evidence": "", "reference": "제공된 근거로는 판단하기 어렵습니다."},
    ]
    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    DATASET = OUTPUT.parent / f"{OUTPUT.name}_input.jsonl"
    DATASET.write_text("".join(json.dumps(x, ensure_ascii=False) + "\n" for x in examples), encoding="utf-8")

## 2. 지표 계산

- **ROUGE-L F1**은 유니코드 정규화 후 연속 문자·숫자 단위로 토큰화하여 LCS F1을 계산합니다.
- **BERTScore F1**은 다국어 mBERT를 사용합니다. IDF 가중치와 baseline 재조정은 끄고 CPU에서 계산합니다.
- **Faithfulness**는 RAGAS 정의의 주장 추출 → DUR 근거 대조를 같은 로컬 채점 모델로 실행하는 별도 구현입니다. 샘플별로 지지된 주장 수를 전체 주장 수로 나눈 뒤 평균합니다.
- **Faithfulness 집계**는 근거 미제공, 빈 응답, 검증 가능한 주장 없음, 채점 오류를 N/A로 처리합니다. 평균과 함께 실제 산출 건수인 `faithfulness_n`을 확인하세요. 빈 응답의 ROUGE-L과 BERTScore F1은 0점입니다.
- **판단 보류 성공률**은 `response_type=판단 보류`인 문항에서 로컬 Judge가 판정합니다. 현재 근거의 한계를 명시하고 적용 여부나 복용 안전성을 근거 없이 단정하지 않으면 성공입니다. 빈 응답은 실패이며, Judge 출력 오류는 집계에서 제외합니다.
- 전체·유형별 표에서 판단 보류 성공 건수(`deferral_success_n`), 채점 건수(`deferral_scored_n`), 대상 건수(`deferral_expected_n`)를 함께 확인하세요. 성공률의 분모는 채점 건수입니다. `response_type`이 없는 기존 데이터는 세 기본 지표를 계산하고 판단 보류 성공률은 N/A로 표시합니다.

두 후보에는 같은 질문·근거·생성 조건을 적용합니다. 최종 Test에서 학습 전후를 비교할 때도 데이터와 평가 설정을 동일하게 맞추세요.

In [ ]:
overall, by_category, per_example = evaluate(
    DATASET, OUTPUT, split=SPLIT, models=CANDIDATES, judge_name=JUDGE_MODEL,
    encoder=BERT_ENCODER, max_new_tokens=MAX_NEW_TOKENS, judge_tokens=JUDGE_TOKENS,
    limit=LIMIT, demo=DEMO, revisions=REVISIONS,
)
print(OUTPUT.resolve())

## 3. 모델별 전체 및 유형별 결과

In [ ]:
display(overall.round(4))
display(Image(filename=str(OUTPUT / "overall.png")))
display(by_category.round(4))
display(Image(filename=str(OUTPUT / "by_category.png")))

## 4. 응답별 결과

표와 응답별 결과는 CSV로, 그래프는 PNG로 저장됩니다. `run_config.json`에는 데이터 해시, 모델 revision, 프롬프트와 실행 설정이 기록됩니다. JSONL에서 추출 주장과 판정을 배열 형태로 확인할 수 있습니다.

In [ ]:
display(per_example[[
    "model", "id", "category", "response_type", "question", "evidence", "prediction", "reference",
    "rouge_l", "bertscore_f1", "faithfulness", "faithfulness_status", "claims", "verdicts",
    "deferral_success", "deferral_status",
]].head(12))